# GENE7 — does morphology add information about the transcriptome?

**The question.** Crispants are mosaic F0s: an injected clutch mixes embryos with effective biallelic
disruption against escapers with little or none. If morphological severity tracks that hidden dose,
then grading embryos by how perturbed they *look* should sharpen the transcriptional signal relative
to a plain crispant/control label.

This notebook condenses the full chain of reasoning. Each detailed analysis lives in a companion
notebook; here the aim is a single coherent account with every number traceable to a file.

**Summary of what was found**

| # | finding | status |
|---|---|---|
| 1 | Supervised morphology axes have genuine predictive power, but **do not outperform** knowing the class label *on the global statistic* | established |
| 2 | The continuous-axis and binary-label transcriptional effects are **strongly aligned** (cosine 0.91); ~17% of the variance is orthogonal — real biology or noise, likely case-dependent | established |
| 2b | Counted in **resolved cell types** rather than effect magnitude, the two are indistinguishable among validated axes (331 vs 338 among validated axes, paired p = 0.78) — Finding 1 is a `T_std` result, not a detection result | established; the `s`-only set has no negative control |
| 3 | Severe crispants trend stronger than escapers, **not significant** on the paired test; more striking is that **wildtype-looking crispants are substantially perturbed** | mixed |
| 4 | Adding an **intra-class** morphology term gives a significant slope in **9 of 35** perturbations, concentrated in those with stronger morphology axes | established (the concentration is suggestive) |
| 5 | That term resolves **67 cell-type effects the label alone misses** (46 of them among validated axes), against **1** in a matched negative control | the headline claim |

---

## 0. Provenance — where every number comes from

**Morphology.** Legacy VAE latents (`z_mu_b_*`, the 80 biological dimensions), assembled per embryo
by `build_master_table()` in `src/morphseq_integration/`. GENE7 is imaging experiment `20250612`.
**14 wells** are dropped by curated image QC (`src/morphseq_integration/excluded_wells.csv`), leaving
**553 embryos**. Design: 3 crispant targets (`atf6`, `ctcf`, `wfs1a/wfs1b`) plus `Control`, at 4
rearing temperatures (24/28/34/35 °C) x 3 collection timepoints (24/30/36 hpf).

**Sequencing.** The v3.1.0 embryo-filtered CDS at
`/net/seahub_zfish/.../portal_inputs/v3.1.0/mcclintock/GENE7/run_1/filter_embryos/embryo_filtered_cds`.
Reduced once by `build_count_table.R` to a cell-type x embryo count matrix: **370 cell types x 651
embryos, 1,894,015 cells**, median 2,341 cells per embryo. Native `cell_type` annotations, not
`cell_type_broad`. The 8.4 GB object is never touched again.

**Overlap.** 535 embryos carry both modalities. 18 embryos have morphology but no cells in the CDS
(§0 table below); 116 sequenced embryos were never imaged.

**Coordinate system.** A GENE7-native 10-component PCA on `z_mu_b_*`, unwhitened; the **first 5**
components are the working subspace. GENE7-native matters — a wildtype-fit basis would pre-filter out
the perturbation directions this analysis is looking for.

| artifact | written by | holds |
|---|---|---|
| `data/gene7_global_scores.csv` | `run_cohort_axes.py` | 553 embryos in the 10D basis |
| `data/lda/contrast_summary.csv` | `run_lda_contrasts.py` | per-contrast axis diagnostics |
| `data/lda/contrast_scores.csv` | `run_lda_contrasts.py` | per-embryo signed distance `s` |
| `data/edger/cell_counts.csv` | `build_count_table.R` | 370 x 651 count matrix |
| `data/edger/contrast_predictors.csv` | `export_edger_inputs.py` | the 7 regression predictors |
| `data/edger/coefficients.csv` | `fit_edger_contrasts.R` | per (contrast, arm, cell type) |
| `data/edger/global_stats.csv` | `fit_edger_contrasts.R` | per (contrast, arm) global statistic |

Companion notebooks: `gene7_lda_contrasts.ipynb` (axis construction and validation),
`gene7_morphospace.ipynb` (PCA and reference spline), `gene7_edger_results.ipynb` (full regression
detail). **Note:** §3b/§3c of `gene7_edger_results.ipynb` predate finding (5) and understate it;
this notebook supersedes them.


In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import binomtest
from IPython.display import display
import matplotlib.pyplot as plt

HERE = Path.cwd()
sys.path.insert(0, str(HERE))
import edger_plots as ep
import lda_plots as lp
ep.use_house_style()
pd.set_option("display.width", 190); pd.set_option("display.max_columns", 40)

LDA = HERE / "data" / "lda"
EDGER = HERE / "data" / "edger"
import gene7_config as cfg                    # shared figure root + exclusion list
FIGURES = cfg.figure_dir("synthesis")        # every figure below is also written here

# EXCLUDED CONTRAST -- defined once in gene7_config.py and shared by every notebook, so the
# figure sets cannot drift apart. atf6 | 34C | 30hpf is dropped for single-point leverage in
# BOTH groups (one crispant at s_z = 4.37 against a next of 0.61, robust z = 11.1; one control
# at 0.99 against a next of -0.06, robust z = 9.8). The call is made on the shape of the
# MORPHOLOGY distribution alone, never on a transcriptional outcome. Applied at load rather
# than in the fitting scripts, so data/edger/*.csv remain a complete record.
EXCLUDED_CONTRASTS = cfg.EXCLUDED_CONTRASTS

axis_summary = pd.read_csv(LDA / "contrast_summary.csv")      # per-contrast axis diagnostics
lda_scores   = pd.read_csv(LDA / "contrast_scores.csv")       # per-embryo signed distance s
coefficients = pd.read_csv(EDGER / "coefficients.csv")        # per (contrast, arm, cell type)
global_stats = pd.read_csv(EDGER / "global_stats.csv")        # per (contrast, arm)
geometry     = pd.read_csv(EDGER / "arm_geometry.csv")
quality      = pd.read_csv(EDGER / "contrast_axis_quality.csv")
predictors   = pd.read_csv(EDGER / "contrast_predictors.csv")
status       = pd.read_csv(EDGER / "contrast_status.csv")
type_index   = pd.read_csv(EDGER / "cell_type_index.csv")

# Audit artifact, §5b only. fit_edger_contrasts.R fits the binary indicator in BOTH designs but
# only writes Model 1's copy to coefficients.csv; check_binary_coefficient.R refits and keeps both.
binary_check   = pd.read_csv(EDGER / "binary_coefficient_check.csv")    # per contrast x cell type
binary_summary = pd.read_csv(EDGER / "binary_coefficient_summary.csv")  # per contrast

_before = quality["contrast"].nunique()
for _name in ("axis_summary", "lda_scores", "coefficients", "global_stats", "geometry",
              "quality", "predictors", "status", "binary_check", "binary_summary"):
    _frame = globals()[_name]
    if "contrast" in _frame.columns:
        globals()[_name] = (_frame[~_frame["contrast"].isin(EXCLUDED_CONTRASTS)]
                            .reset_index(drop=True))

N = quality["contrast"].nunique()
N_USABLE = int(quality["usable"].sum())
print(f"excluded {_before - N} contrast(s) for single-point leverage: "
      f"{', '.join(EXCLUDED_CONTRASTS)}")
print(f"{N} contrasts ({N_USABLE} with a usable axis) | "
      f"{predictors['sample'].nunique()} embryos with morphology | {len(type_index)} cell types "
      f"| {len(coefficients):,} cell-type tests")
print(f"figures -> {FIGURES}")


In [ ]:
print("embryos with morphology but no cells in the CDS, by contrast:")
display(status.loc[status["n_dropped"] > 0,
                   ["contrast", "n_embryos", "n_dropped", "n_cell_types"]]
        .sort_values("n_dropped", ascending=False).head(8))
print(f"\ncompositional risk check — largest cell type is "
      f"{type_index['mean_fraction'].max():.1%} of cells, top 5 sum to "
      f"{type_index['mean_fraction'].nlargest(5).sum():.1%}")
print("(a dominant cell type would drag every other coefficient and fill the results with "
      "reciprocal artefacts; that failure mode is not in play here)")


---

## 1. Supervised morphology axes work — but do not beat the class label

**Construction.** For each of 35 retained (target x temperature x timepoint) cells, a shrunken-LDA hyperplane
separates that crispant group from its **stage- and temperature-matched controls** in the 5D
subspace. Each embryo's signed distance normal to that plane is `s`. Ledoit-Wolf shrinkage rather
than plain LDA because 11-vs-12 in 5D under-determines the covariance; *logistic* regression rather
than LDA would have diverged outright, since 24 points in 5D are all but certain to be separable.

**Validation.** Every axis was assessed before any sequencing data was touched — leave-one-out AUC
with a label-permutation null, bootstrap stability of both the direction and the resulting embryo
*ordering*, and the angle to a control-derived developmental-stage axis. 16 of 35 axes clear both
gates. Details in `gene7_lda_contrasts.ipynb`.


In [ ]:
gates = pd.DataFrame({
    "permutation q<0.10 (LOO-AUC)": [(axis_summary["q_auc"] < 0.10).sum()],
    "ordinally stable vs its own random floor": [axis_summary["rank_stable"].sum()],
    "usable (both gates)": [axis_summary["usable"].sum()],
    "stage-confounded (angle < 35 deg)": [axis_summary["stage_confounded"].sum()],
}, index=[f"of {len(axis_summary)}"]).T
display(gates)
print(f"median LOO-AUC {axis_summary['loo_auc'].median():.3f} "
      f"(usable only: {axis_summary.loc[axis_summary['usable'],'loo_auc'].median():.3f})")


In [ ]:
figure = ep.head_to_head(global_stats, quality)
ep.save_inline(figure, FIGURES / "1_head_to_head")


**The head-to-head.** Three one-predictor models per contrast, all z-scored and all carrying one
degree of freedom so they are directly comparable: `binary` (the indicator), `s` (the signed
distance), and `hinge` = `max(s − control mean, 0)`. The statistic is `T_std`, a
permutation-calibrated measure of how far the whole cell-type composition moved from its null.

`T_std` is **scale-invariant** — rescaling a predictor rescales the score and its null SD together —
so the comparison cannot be won by `s` merely having more range than a 0/1 indicator.


In [ ]:
wide = global_stats.pivot(index="contrast", columns="arm", values="t_std").join(
    quality.set_index("contrast")[["usable", "loo_auc"]])
rows = []
for label, subset in ((f"all {N}", wide), (f"usable axes ({N_USABLE})", wide[wide["usable"]])):
    for arm in ("s_z", "hinge_z"):
        delta = (subset[arm] - subset["binary_z"]).dropna()
        rows.append({"set": label, "arm": arm, "wins": int((delta > 0).sum()), "n": len(delta),
                     "median_delta": delta.median(), "wilcoxon_p": stats.wilcoxon(delta)[1]})
display(pd.DataFrame(rows).round(4))


**Finding 1.** On `T_std`, the binary indicator wins in 33 of 35 contrasts (14 of 16 among validated
axes), median ΔT_std ≈ −2.7. Restricting to good axes does not rescue it, and neither does the hinge.
As a *replacement* for the class label, the morphology score loses on this statistic — decisively and
reproducibly.

**Scope that claim to `T_std`.** §2b re-runs the same comparison counted in *resolved cell types* and
finds no difference at all among the validated axes (paired p = 0.78). `T_std` = Σz² is driven by the
magnitude of the largest effects; a hit count saturates at the FDR threshold. Both are true of the
same fits, and the distinction matters for how the result should be stated.

**`s` and `hinge` are the same arm in practice.** Collapsing the controls to zero — the obvious fix
for control-side dispersion diluting the loadings — changes nothing: paired Wilcoxon on `T_std` gives
**p = 0.84** across all 35 (median Δ +0.02) and p = 0.43 among the usable 16, with cosine to binary
0.913 vs 0.911 and magnitude ratio 0.891 vs 0.877. Everything downstream uses `s_z`; `hinge_z` is
still fit and written out, but is not carried through the narrative.

Worth being clear about what this does and does not say: it says a graded morphological score is a
worse single predictor than knowing which embryos were injected. It says nothing yet about whether
morphology adds anything *on top of* that knowledge. That is §4.

---

## 2. The two effects are strongly aligned, with ~17% of the variance elsewhere


In [ ]:
figure = ep.geometry_plot(geometry, quality)
ep.save_inline(figure, FIGURES / "2_geometry")


In [ ]:
merged = geometry.merge(quality[["contrast", "usable"]], on="contrast")
rows = []
for arm in ("s_z", "hinge_z"):
    block = merged[(merged["arm"] == arm) & merged["usable"]]
    cosine = block["cosine_to_binary"].median()
    rows.append({
        "arm": arm,
        "median_cosine_to_binary": cosine,
        "shared_variance (cos^2)": cosine ** 2,
        "orthogonal_variance (1-cos^2)": 1 - cosine ** 2,
        "orthogonal_fraction_of_length": (block["orthogonal_component"] / block["norm"]).median(),
        "median_magnitude_ratio": (block["norm"] / block["norm_binary"]).median(),
    })
pd.DataFrame(rows).round(3)


### 2b. The same comparison, counted in resolved cell types

The panels above compare **coefficient vectors** — direction and length in ~370-dimensional score
space. That is the right object for asking *whether the two arms found the same biology*, and the
answer was yes (cosine 0.91). It is not the object a reader asks about, which is **how many cell
types each arm actually resolves, and how much those sets overlap.**

The two measures can disagree, and here they do. `T_std` is a sum of squared standardised scores, so
it is dominated by the *magnitude* of the strongest effects. A hit count saturates — a cell type at
q = 10⁻³⁶ counts exactly as much as one at q = 0.09. An arm that produces a few enormous effects and
an arm that produces many moderate ones can differ hugely on `T_std` and not at all on hit count.

Three disjoint categories per contrast: **binary-only**, **shared**, **`s`-only**. Only `s_z` is
shown; §1's head-to-head established that `hinge_z` is indistinguishable from it (paired Wilcoxon
p = 0.84 across all 35, p = 0.43 among usable), so carrying both adds nothing.

**One caveat up front, because it is the difference between this and §5.** The within-crispant slope
has a matched negative control — the within-control slope, same model, same embryos — that bounds how
much of its yield is FDR noise. **Nothing bounds the `s`-only set here.** The comparison is fair
(same one-predictor design, same single df, same cell-type filter, same fixed dispersion, same BH
family, no tuning) but fair is not the same as verified.


In [ ]:
figure, overlap = ep.hit_overlap_plot(coefficients, quality, arm="s_z")
ep.save_inline(figure, FIGURES / "2b_hit_overlap")


In [ ]:
# Bar-only version — the middle panel plus a pooled summary, sized for a slide.
# render_slide_figures.py writes this and the two unsupervised counterparts to
# figures/slides/ on identical axes, so the three can sit side by side.
figure, _ = ep.hit_overlap_bars(coefficients, quality, arm="s_z",
                                arm_label="$s$", exclude=EXCLUDED_CONTRASTS)
ep.save_inline(figure, FIGURES / "2b_hit_overlap_bars")


In [ ]:
# Same comparison, all 35 contrasts, but the 19 whose morphology axis failed the §1 validity
# gate are drawn translucent. Keeps full coverage visible without giving unvalidated axes equal
# visual weight: of the 115 s-only cell types, 79 come from validated axes and 36 from faded ones.
figure, faded = ep.hit_overlap_bars(coefficients, quality, arm="s_z", arm_label="$s$",
                                    exclude=EXCLUDED_CONTRASTS, fade_unvalidated=True)
ep.save_inline(figure, FIGURES / "2c_hit_overlap_bars_faded")
print("s-only cell types  validated %d | unvalidated (faded) %d"
      % (int(faded.loc[faded["usable"], "arm_only"].sum()),
         int(faded.loc[~faded["usable"], "arm_only"].sum())))

In [ ]:
rows = []
for label, block in ((f"all {N}", overlap), (f"usable axes ({N_USABLE})", overlap[overlap["usable"]])):
    rows.append({
        "set": label, "n": len(block),
        "binary_only": int(block["binary_only"].sum()),
        "shared": int(block["shared"].sum()),
        "s_only": int(block["morph_only"].sum()),
        "binary_total": int(block["binary_total"].sum()),
        "s_total": int(block["arm_total"].sum()),
        "median_jaccard": block["jaccard"].median(),
        "median_recovery": block["recovery"].median(),
    })
display(pd.DataFrame(rows).round(3))

# The comparison that matters: paired per contrast, on BOTH metrics. They give opposite answers,
# and the reason is that T_std weights by magnitude while a hit count saturates at the threshold.
tw = global_stats.pivot(index="contrast", columns="arm", values="t_std")
counts = overlap.set_index("contrast")
for label, idx in ((f"all {N}", counts.index),
                   ("usable axes", counts.index[counts["usable"]])):
    d_hits = (counts.loc[idx, "arm_total"] - counts.loc[idx, "binary_total"]).dropna()
    d_tstd = (tw.loc[idx, "s_z"] - tw.loc[idx, "binary_z"]).dropna()
    print(f"{label:12s} HITS   s beats binary {int((d_hits > 0).sum()):2d}/{len(d_hits)} "
          f"(ties {int((d_hits == 0).sum()):2d}), median {d_hits.median():+.1f}, "
          f"Wilcoxon p = {stats.wilcoxon(d_hits)[1]:.3f}")
    print(f"{'':12s} T_std  s beats binary {int((d_tstd > 0).sum()):2d}/{len(d_tstd)}, "
          f"median {d_tstd.median():+.2f}, Wilcoxon p = {stats.wilcoxon(d_tstd)[1]:.4f}")

# Is the usable-subset parity carried by one contrast? Check before quoting the totals.
u = counts[counts["usable"]]
delta = (u["arm_total"] - u["binary_total"]).sort_values()
top = delta.abs().idxmax()
print(f"\nusable subset, sum of (s - binary) hit counts: {delta.sum():+.0f}")
print(f"  largest single contributor: {delta[top]:+.0f}  ({top})")
print(f"  sum with it removed:        {delta.drop(top).sum():+.0f}   <- the totals are NOT robust; "
      f"the paired test is the honest summary")


**Finding 2.** Coefficient vectors from the `s` arm and the `binary` arm point in nearly the same
direction — median cosine **0.91** across validated contrasts. Morphology is not finding a different
transcriptional axis; it is finding the same one.

The residual deserves care because two framings of it differ by a factor of two, and both are
correct:

- **~17% of the variance** is orthogonal to the binary contrast (`1 − cos²`)
- **~41% of the vector length** is orthogonal (`sqrt(1 − cos²)`)

The variance framing is the more conservative and the one quoted above. Whether that residual is
real biology or noise is **not resolved here**, and is most likely case-dependent — §5 shows that in
some contrasts it corresponds to genuinely new cell types, while in most it does not.

Magnitude ratio is 0.89: the same signal, slightly attenuated.

**Finding 2b, and it qualifies Finding 1.** Counted in *resolved cell types* rather than vector
magnitude, the two arms are far closer than `T_std` suggests — and among the validated axes they are
statistically indistinguishable:

| | all 35 | usable axes (16) |
|---|---|---|
| binary-only | 189 | 72 |
| shared | 343 | 259 |
| **`s`-only** | **115** | **79** |
| totals, binary vs `s` | 532 vs 458 | 331 vs **338** |
| median Jaccard | 0.42 | **0.61** |
| median recovery of binary's hits | 57% | **80%** |
| paired test, hit counts | s wins 11/35, p = 0.051 | s wins 7/16, **p = 0.78** |
| paired test, `T_std` | s wins 2/35, **p < 1e-4** | s wins 2/16, **p = 0.002** |

The last two rows are the point. **On `T_std` the binary indicator wins decisively; on cell types
resolved it does not win at all within the gated subset.** Both statements are true of the same fits.
`T_std` = Σz² is driven by the magnitude of the largest effects — binary q-values reach 10⁻³⁶ — while
a hit count saturates at the FDR threshold. The class label produces much more extreme individual
effects while resolving about the same number of cell types.

So Finding 1's "decisively and reproducibly" should be read as scoped to the global statistic. The
morphology score is a worse *estimator of effect magnitude* for the group difference. It is not a
worse *detector of affected cell types*, once the axis is one that passed validation.

**Do not quote the 331-vs-338 as a win.** That +7 is one contrast: `atf6 | 34C | 36hpf` contributes
+21 by itself, and the sum without it is −14. The paired Wilcoxon at p = 0.78 is the honest summary —
a wash, not an advantage. And the 79 `s`-only cell types have **no negative control**, unlike the 67
in §5; at q < 0.10 roughly 34 of the 338 calls are expected false, and nothing here says which.

---

## 3. Escapers are perturbed; the severity split is directionally right but weak

Crispants are split at the **upper edge of the observed control range**: an "escaper" is an injected
embryo whose `s` falls entirely inside the controls' span. Note this is deliberately the most
permissive definition available, and the category is still thin — for a structural reason. A contrast
has a trustworthy axis precisely when the groups separate cleanly, which is when few crispants land
in control territory. **Escaper-rich and axis-trustworthy are in direct tension.** 21 of 35 contrasts
have at least 4 embryos in both cells.

Embryos are never *dropped* on the basis of `s`. Deleting the wildtype-looking crispants would remove
observations conditional on a label-derived quantity — precisely those nearest the decision boundary,
whose transcriptomes are presumably also the most control-like — and would mechanically widen the
group gap.


In [ ]:
figure = ep.three_level_plot(global_stats, quality)
ep.save_inline(figure, FIGURES / "3_three_level")


In [ ]:
rows = []
for arm in ("escaper_vs_control", "severe_vs_control", "severe_vs_escaper", "binary_z"):
    block = global_stats[global_stats["arm"] == arm]
    rows.append({"arm": arm, "n_contrasts": len(block),
                 "median_T_std": block["t_std"].median(),
                 "p_global_lt_05": int((block["p_global"] < 0.05).sum()),
                 "median_hits_q10": block["n_hits_q10"].median()})
display(pd.DataFrame(rows).round(3))

paired = (global_stats[global_stats["arm"] == "escaper_vs_control"][["contrast", "t_std"]]
          .merge(global_stats[global_stats["arm"] == "severe_vs_control"][["contrast", "t_std"]],
                 on="contrast", suffixes=("_escaper", "_severe")))
gap = paired["t_std_severe"] - paired["t_std_escaper"]
print(f"INDIRECT paired (severe - escaper, each vs control), n={len(gap)}: "
      f"median {gap.median():+.2f}, severe larger in {int((gap>0).sum())}/{len(gap)}, "
      f"Wilcoxon p = {stats.wilcoxon(gap)[1]:.3f}")
direct = global_stats[global_stats["arm"] == "severe_vs_escaper"]
k, n = int((direct["p_global"] < 0.05).sum()), len(direct)
print(f"DIRECT severe vs escaper (controls dropped), n={n}: {k}/{n} at p<0.05, "
      f"chance {0.05*n:.1f}, binomial p = {binomtest(k, n, 0.05, alternative='greater').pvalue:.4f}")


**Finding 3, two halves.**

*The severity split.* Severe crispants trend stronger than escapers, but the **paired** comparison —
the right way to compare, since the two sub-contrasts are estimable on different contrast sets — gives
a median gap of +1.37 in only 12 of 21 contrasts, Wilcoxon **p = 0.11**. Not significant. The
**direct** contrast (severe vs escaper, controls dropped, both groups injected so batch and injection
efficiency cancel) does clear chance at 4/21 versus 1.1 expected, **binomial p = 0.019** — but with
median `T_std` +0.71 and zero cell types surviving FDR. So: directionally right, weakly supported,
not a strong result either way.

*The more striking half.* **Wildtype-looking crispants are substantially transcriptionally
perturbed** — median `T_std` +3.9, significant in 20 of 26 estimable contrasts, median 6.5 cell types
past FDR. They carry roughly **70%** of the severe group's composition shift while being
morphologically indistinguishable from controls.

That single fact explains §1 and §2. The mosaic-dose premise required escapers to be near-unperturbed
so that appearance would recover dose. They are not. Morphological severity is a **weak and
saturating** proxy for molecular severity, which is why substituting it for a clean label attenuates
rather than sharpens.

---

## 4. Intra-class morphology *does* add information on top of the label

§1 asked whether `s` can **replace** the label. The different question is whether the graded part
adds anything **beyond** it.

`s` is split into the part that *is* the label and the part that is not:
`s_within = s − (own group's mean of s)`. That column has zero mean inside both groups, hence is
orthogonal to the binary indicator as constructed (|corr| < 1e-16 before the CDS join; see §5b — the join breaks it slightly, to |corr| <= 0.126). Without the centering the
two predictors are near-collinear — `s` was built to separate the groups — and both coefficients
would carry inflated standard errors.

Slopes are fit **separately per group**, which buys a matched negative control: controls also vary
along `s`, and if that variation is measurement noise the control slope should sit at chance. If both
slopes moved together, that would indicate a shared nuisance — developmental stage being the obvious
suspect.


### DACT table — control vs escaper vs severe, per cell type

Finding 3 above is stated at the **global** level (`T_std`, hit counts per contrast). This assembles
the same three sub-contrasts into a per-cell-type **DACT** table, in the lab's usual sense of
"differentially abundant cell type": one row per (contrast, cell type), the log fold-change and
q-value for each of the three comparisons, and a flag for which of them call it.

Two differences from the Hooke DACT convention worth stating, because the numbers are not
interchangeable with a `contrast_abundance_tbl.tsv`:

- these are **edgeR NB-GLM** contrasts, not Hooke/PLN, so the gate is edgeR's BH `q_value`, not
  Hooke's `delta_q_value`;
- the project-wide threshold here is **q < 0.10**, not the 0.05 used in the GENE14 DACT work. Both
  are tabulated below so the stricter reading is available.

`severe_vs_escaper` is the informative column: both groups were injected, so injection efficiency,
handling and batch cancel, and it is the direct test of whether morphological severity stratifies
the transcriptome.

In [ ]:
DACT_ARMS = ["escaper_vs_control", "severe_vs_control", "severe_vs_escaper"]
DACT_Q = 0.10                                  # project-wide gate; 0.05 also reported below

# One row per (contrast, cell type); one logFC/q column pair per sub-contrast. An outer merge, so a
# cell type tested in only some of the three arms is kept with NaN rather than silently dropped --
# the three arms run on different embryo subsets and therefore different filterByExpr sets.
dact = None
for arm in DACT_ARMS:
    block = (coefficients.loc[coefficients["arm"] == arm,
                              ["contrast", "cell_type", "logFC", "q_value"]]
             .rename(columns={"logFC": f"logfc_{arm}", "q_value": f"q_{arm}"}))
    dact = block if dact is None else dact.merge(block, on=["contrast", "cell_type"], how="outer")

dact = cfg.drop_excluded(dact)
for arm in DACT_ARMS:
    dact[f"dact_{arm}"] = dact[f"q_{arm}"] < DACT_Q
dact["n_arms_calling"] = dact[[f"dact_{a}" for a in DACT_ARMS]].sum(axis=1)

# Which pattern of calls a cell type shows. "escaper only" is the one that matters for the second
# half of Finding 3: a cell type perturbed in wildtype-LOOKING crispants but not in severe ones.
def dact_class(row):
    escaper, severe = row["dact_escaper_vs_control"], row["dact_severe_vs_control"]
    if escaper and severe:
        return "both vs control"
    if severe:
        return "severe only"
    if escaper:
        return "escaper only"
    return "neither vs control"

dact["dact_class"] = dact.apply(dact_class, axis=1)
dact = dact.sort_values(["contrast", "q_severe_vs_control"]).reset_index(drop=True)

target = EDGER / "dact_escaper_severe.csv"
dact.to_csv(target, index=False)
print(f"wrote {target}  ({len(dact)} contrast x cell-type rows, "
      f"{dact['contrast'].nunique()} contrasts)\n")

for threshold in (0.10, 0.05):
    counts = {a: int((dact[f"q_{a}"] < threshold).sum()) for a in DACT_ARMS}
    print(f"DACTs at q < {threshold:.2f}:  " +
          "   ".join(f"{a} {n}" for a, n in counts.items()))
print()
display(dact["dact_class"].value_counts().rename("cell types").to_frame())

In [ ]:
from matplotlib import gridspec

CATEGORIES = [("escaper only", "#8FA9C4"), ("both vs control", "0.62"),
              ("severe only", "#C0392B")]

counts = (dact.groupby(["contrast", "dact_class"]).size().unstack(fill_value=0)
          .reindex(columns=[name for name, _ in CATEGORIES], fill_value=0).reset_index())
counts = ep.order_by_design(counts)

figure = plt.figure(figsize=(10.4, 7.0))
grid = gridspec.GridSpec(2, 1, height_ratios=[len(counts), 2.2], hspace=0.34, figure=figure)

axis = figure.add_subplot(grid[0, 0])
positions = np.arange(len(counts))
left = np.zeros(len(counts))
for name, colour in CATEGORIES:
    axis.barh(positions, counts[name], height=0.74, left=left, color=colour, zorder=3,
              label=f"{name}  ({int(counts[name].sum())})")
    left += counts[name].to_numpy()
axis.set_yticks(positions)
axis.set_yticklabels(counts["contrast"], fontsize=7.4)
axis.invert_yaxis()
axis.set_xlabel(f"cell types called at q < {DACT_Q:g}", fontsize=9)
axis.tick_params(labelbottom=True, labelsize=8.5)
axis.legend(fontsize=9.2, loc="lower right")
axis.set_title("Differentially abundant cell types: escaper and severe crispants vs control",
               fontsize=11.5, pad=10)

pooled = figure.add_subplot(grid[1, 0])
left = 0.0
for name, colour in CATEGORIES:
    width = float(counts[name].sum())
    pooled.barh([0], [width], left=[left], height=0.5, color=colour, edgecolor="white",
                linewidth=1.0, zorder=3)
    if width >= 40:
        pooled.text(left + width / 2, 0, f"{int(width)}", ha="center", va="center",
                    fontsize=9.5, color="white" if colour == "#C0392B" else "0.15", zorder=4)
    left += width
pooled.barh([1], [int((dact["q_severe_vs_escaper"] < DACT_Q).sum())], height=0.5,
            color="#7D3C98", zorder=3)
pooled.set_yticks([0, 1])
pooled.set_yticklabels([f"pooled ({len(counts)} contrasts)", "severe vs escaper (direct)"],
                       fontsize=9)
pooled.invert_yaxis()
pooled.set_xlabel("cell types  (pooled — note the separate scale)")
pooled.spines["left"].set_visible(False)
pooled.tick_params(axis="y", length=0)

ep.save_inline(figure, FIGURES / "3b_dact_escaper_severe")

In [ ]:
rows = []
for arm, name in (("s_within_crispant", "within-CRISPANT (the dose effect)"),
                  ("s_within_control", "within-CONTROL (negative control)")):
    b = global_stats[global_stats["arm"] == arm]
    k, n = int((b["p_global"] < 0.05).sum()), len(b)
    rows.append({"slope": name, "hits_p<0.05": f"{k}/{n}", "expected_by_chance": round(0.05*n, 1),
                 "binomial_p": binomtest(k, n, 0.05, alternative="greater").pvalue,
                 "median_T_std": b["t_std"].median()})
display(pd.DataFrame(rows).round(4))

pv = global_stats.pivot(index="contrast", columns="arm", values="p_global")
print(f"paired Wilcoxon on the two slopes' p-values: "
      f"p = {stats.wilcoxon(pv['s_within_crispant'], pv['s_within_control'])[1]:.4f}")
print(f"crispant slope more significant than its own control in "
      f"{int((pv['s_within_crispant'] < pv['s_within_control']).sum())}/{len(pv)} contrasts")


**Finding 4a.** The within-crispant slope is significant in **9 of 35** contrasts where chance
gives 1.8 — binomial p ≈ 1e-5 — while the matched negative control sits at the chance rate. Paired,
the crispant slope beats its own control in 26 of 35 contrasts (Wilcoxon p = 0.002).

*(The count moves between 9 and 10 across runs: contrasts sitting near p = 0.05 flip with the
permutation draw at 2000 permutations. The binomial p is ~1e-5 either way.)*

### Which measure of "morphological phenotype" predicts where the slope fires?


In [ ]:
figure = ep.metric_comparison(global_stats, quality)
ep.save_inline(figure, FIGURES / "4_metric_comparison")


In [ ]:
cri = global_stats[global_stats["arm"] == "s_within_crispant"].set_index("contrast")
ctl = global_stats[global_stats["arm"] == "s_within_control"].set_index("contrast")
qq = quality.set_index("contrast")
rows = []
for metric, label in [("log_sd_ratio", "crispant/control spread"),
                      ("rank_rho_median", "ordinal stability"),
                      ("top_retention_median", "top-tercile retention"),
                      ("separation", "Mahalanobis separation"),
                      ("loo_auc", "LOO-AUC"),
                      ("boot_angle_p95", "bootstrap angle p95 (inverted)")]:
    d = cri[["p_global"]].join(qq[[metric]]).dropna()
    c = ctl[["p_global"]].join(qq[[metric]]).dropna()
    r = stats.spearmanr(d[metric], -np.log10(d["p_global"]))
    rc = stats.spearmanr(c[metric], -np.log10(c["p_global"]))
    rows.append({"metric": label, "rho_vs_slope_significance": r.statistic, "p": r.pvalue,
                 "neg_control_rho": rc.statistic, "neg_control_p": rc.pvalue})
pd.DataFrame(rows).round(3)


**Finding 4b.** The slope fires preferentially where the morphological phenotype is stronger, and the
best predictor is **crispant/control spread** (rho +0.42, p = 0.011), with ordinal stability second
(+0.34, p = 0.044). Mahalanobis separation and LOO-AUC are middling (p ≈ 0.11, 0.12); bootstrap angle
is worst.

Two things about how to read that. **The winner is mechanistically the right quantity**, not a fluke
of searching: separation and AUC measure how far crispants sit *from controls*, whereas
crispant/control spread measures how heterogeneous the crispants are *among themselves* — and a
within-group dose analysis can only exploit within-group variation. A cleanly separated but internally
uniform crispant group offers no gradient to detect. **But six metrics were compared**, and Bonferroni
over six puts the best at p ≈ 0.066, so the ranking among them should not be over-read.

Every negative control in that table is flat or slightly negative, which is what rules out "some
contrasts are simply cleaner than others".

So morphology adds information in a **supplementary and perturbation-dependent** capacity: it does
not help everywhere, it helps where there is graded morphology to exploit.

---

## 5. The gain, in cell types

`T_std` is a global statistic. This section asks the concrete question: **does the morphology term
resolve cell types the label alone misses?** Counted by set membership per contrast, not by comparing
totals — the distribution is zero-inflated and heavily skewed, so a median would hide it entirely.

The baseline is not thin: the binary indicator already resolves a median of **11** cell types per
contrast (**16** among validated axes) out of ~50 tested, with only 2 of 35 contrasts finding nothing.
There is plenty of resolution to add to.


In [ ]:
# Gains over the plain binary contrast, split by HOW they arrive. Four figures: binary-only
# then binary+morph, for all contrasts and for validated axes only. Same rows, ordering and
# x-limits within each pair, so they render as a build.
_frames = {}
for _usable, _tag in ((False, "all"), (True, "gated")):
    for _layers, _suffix in (("binary", "1_binary_only"), ("both", "2_binary_plus_morph")):
        _fig, _frame = ep.slope_gain_bars(binary_check, quality, usable_only=_usable,
                                          layers=_layers)
        ep.save_inline(_fig, FIGURES / f"5_gain_{_tag}_{_suffix}")
        _frames[_tag] = _frame
gains, gains_gated = _frames["all"], _frames["gated"]   # `gains` must be the FULL set

# the matched negative control for the INDIRECT channel: two designs with one extra column
# each, differing only in which group carries the gradient
b = binary_summary
print("indirect additions to the binary coefficient, matched 3-parameter designs:")
for col_gain, col_lost, name in (("indirect_m4_crispslope", "lost_m4_crispslope",
                                  "~ binary + s_within_CRISPANT  (the real thing)"),
                                 ("indirect_m3_ctrlslope", "lost_m3_ctrlslope",
                                  "~ binary + s_within_CONTROL   (negative control)")):
    print(f"  {name:46s} +{int(b[col_gain].sum()):3d}  -{int(b[col_lost].sum()):2d}  "
          f"net {int(b[col_gain].sum() - b[col_lost].sum()):+3d}  "
          f"in {int((b[col_gain] > 0).sum())} contrasts")
_d = b["indirect_m4_crispslope"] - b["indirect_m3_ctrlslope"]
print(f"  paired: crispant > control in {int((_d > 0).sum())}/{len(_d)}, "
      f"never the reverse ({int((_d < 0).sum())}), Wilcoxon p = {stats.wilcoxon(_d)[1]:.4f}")


In [ ]:
figure = ep.volcano_within(coefficients)
ep.save_inline(figure, FIGURES / "5_volcano_within")


### 5b. Are the new cell types *new*, or just marginal?

The counts above are set differences, which treat "found" as binary. That hides the thing an auditor
needs: **how close did the binary contrast come** on the cell types the slope claims? A hit whose
binary q is 0.11 is a rounding decision. A hit whose binary q is 0.9 is a genuinely different signal.

Two comparators, because the answer depends on which binary fit is meant and the gap is not
negligible:

- **Model 1**, `~ binary` on its own — the analysis anyone would actually run, and the baseline the
  67 is defined against.
- **Model 2**, the binary column of the *same* four-column design the slope comes from. Holding the
  design fixed isolates the coefficient from the model, so anything new here is new because of the
  gradient rather than because the comparator was refitted.

Model 2's binary coefficient is fitted by `fit_edger_contrasts.R` and then discarded — only coefs 3
and 4 are written out. `check_binary_coefficient.R` refits both designs under identical conditions
(same offset, same `filterByExpr` on the binary design, same fixed dispersion) and keeps all three
coefficients.


In [ ]:
figure, slope_hits = ep.slope_versus_binary_q(binary_check)
ep.save_inline(figure, FIGURES / "5b_slope_vs_binary_q")


In [ ]:
# Does the four-column design cost the binary contrast anything? The design comment in
# fit_edger_contrasts.R says no, on the grounds that s_within is orthogonal to binary. That
# orthogonality is asserted in export_edger_inputs.py, which runs BEFORE the join to the CDS.
b = binary_summary
print(f"binary hits q<0.10   Model 1: {int(b['hits_binary_m1'].sum())}   "
      f"Model 2: {int(b['hits_binary_m2'].sum())}   "
      f"(shared {int(b['shared_m1_m2'].sum())}, lost {int(b['lost_in_m2'].sum())}, "
      f"gained {int(b['gained_in_m2'].sum())})")
print(f"  retention of the Model 1 set : "
      f"{b['shared_m1_m2'].sum() / b['hits_binary_m1'].sum():.1%}")
print(f"  logFC correlation M1 vs M2   : median {b['logfc_cor_m1_m2'].median():.4f} "
      f"(min {b['logfc_cor_m1_m2'].min():.4f})")
print(f"  residual df                  : M1 {b['df_model1'].min()}-{b['df_model1'].max()}, "
      f"M2 {b['df_model2'].min()}-{b['df_model2'].max()}")
print()
print("orthogonality of the added columns to the binary column, IN THE FITTED DESIGN:")
print(f"  max |corr| {b['max_abs_cor_to_binary'].max():.3f}   "
      f"median {b['max_abs_cor_to_binary'].median():.2e}   "
      f"above 0.05 in {(b['max_abs_cor_to_binary'] > 0.05).sum()}/{len(b)} contrasts")
worst = b.nlargest(3, "max_abs_cor_to_binary")
display(worst[["contrast", "n_embryos", "max_abs_cor_to_binary", "hits_binary_m1",
               "hits_binary_m2", "hits_slope", "new_vs_m1", "new_vs_m2"]].round(3))

print("how deep in the binary's null the NEW cell types sit:")
for column, name in (("q_binary_m1", "Model 1"), ("q_binary_m2", "Model 2")):
    new = slope_hits[slope_hits[column] >= 0.10]
    print(f"  {name}: {len(new)} new | median binary q {new[column].median():.3f} | "
          f"{(new[column] >= 0.5).sum()} of them at q >= 0.5")


**Finding 5b — the new cell types are mostly not marginal, and 67 is the generous count.**

| of the slope's hits | all 35: vs Model 1 | all 35: vs Model 2 | usable 16: vs Model 1 | usable 16: vs Model 2 |
|---|---|---|---|---|
| total slope hits | 165 | 165 | 128 | 128 |
| also a binary hit, q < 0.10 | 98 | 112 | 82 | 93 |
| near miss, 0.10 ≤ q < 0.25 | 25 | 17 | 16 | 10 |
| no binary signal, q ≥ 0.25 | **42** | **36** | **30** | **25** |
| **NEW (total)** | **67** | **53** | **46** | **35** |
| median binary q among the NEW | 0.362 | 0.360 | 0.416 | 0.537 |

**They are not rounding decisions.** Median binary q among the 67 new cell types is **0.36**, and 25
of them sit at q ≥ 0.5 — the middle of the binary contrast's null. Only 25 of 67 are near misses. The
gradient is finding effects the group comparison gives no hint of, not nudging borderline ones over
the line. Inside the gated subset the separation is cleaner still: median binary q among the new hits
rises to **0.42** (0.54 under the strict comparator), and the near-miss fraction falls from 37% to 35%
(and to 29% strict).

**But 67 is the generous count; 53 is the strict one.** 14 of the 67 are cell types that Model 1's
binary coefficient misses and Model 2's binary coefficient catches — so they are new relative to the
*standard analysis*, but not attributable to the gradient specifically. Scored against the binary
coefficient of its own design, the slope contributes **53** cell types no group contrast finds either
way (**35** within the gated subset). Both numbers answer real questions; the strict pair is what
survives the harshest reading, and 35-against-0 still carries the claim.

**The added columns do not cost the group contrast anything.** Model 2's binary coefficient retains
**98.7%** of Model 1's hits (525 of 532, 7 lost) and gains 32 more, for 557 against 532 — the two
extra parameters cost 2 residual df but the slopes absorb enough within-group variance to more than
repay it. logFC correlation is 0.998 (min 0.967) with 100% sign agreement on the shared hits. Nothing
was traded away to fit the gradient.

**One correction to §4's phrasing.** §4 states that `s_within` is orthogonal to the binary indicator
"at |corr| < 1e-16". That is verified in `export_edger_inputs.py`, which runs *before* the join to the
CDS — and the 22 embryos dropped at that join for having no cells break the exact group-mean centring
the guarantee rests on. In the design actually fitted, |corr| reaches **0.126** (`ctcf | 24C | 24hpf`,
which drops 3 of 23 embryos) and exceeds 0.05 in 2 of 35 contrasts. Median is still ~0, and this is
what the 7 lost / 32 gained hits are made of, but the correct word is *near*-orthogonal.

**Read the vertical spread too.** The slope's q-values top out at 10⁻³·⁸, while binary hits reach
10⁻³⁶. These are different coefficients answering different questions, not two estimates of one
quantity — but it is worth being explicit that every slope hit is a modest-confidence call sitting
close to the FDR threshold, and their credibility rests on the negative control rather than on
individual q-values.


In [ ]:
summary = pd.DataFrame([{
    "channel": name,
    "all_hits": int(gains[col].sum()),
    "all_contrasts": int((gains[col] > 0).sum()),
    "gated_hits": int(gains.loc[gains["usable"], col].sum()),
    "gated_contrasts": int((gains.loc[gains["usable"], col] > 0).sum()),
    "max_in_one_contrast": int(gains[col].max()),
} for name, col in (("baseline — simple binary contrast", "baseline"),
                    ("indirect — group contrast sharpened", "indirect"),
                    ("direct — gradient only", "direct"),
                    ("lost from the binary coefficient", "lost"),
                    ("within-CONTROL slope (neg. control)", "control_slope"))])
display(summary)

# Rates, not totals. The two subsets differ in how many tests they run and in how strong the
# perturbations are, so raw counts overstate the gating effect. Normalising each channel by
# its own subset's binary baseline is the stricter comparison.
for label, block in (("usable axes", gains[gains["usable"]]),
                     ("non-usable", gains[~gains["usable"]])):
    base, tested = block["baseline"].sum(), block["n_tested"].sum()
    print(f"{label:12s} n={len(block):2d} | {int(tested):4d} tests | "
          f"baseline {base / tested:5.1%} of tests | "
          f"indirect +{block['indirect'].sum() / base:4.1%} of baseline | "
          f"direct +{block['direct'].sum() / base:4.1%} of baseline")

m = (coefficients[coefficients["arm"] == "s_within_crispant"]
     [["contrast", "cell_type", "logFC", "q_value"]]
     .merge(coefficients[coefficients["arm"] == "binary_z"][["contrast", "cell_type", "logFC"]],
            on=["contrast", "cell_type"], suffixes=("_within", "_binary")))
usable_set = set(quality.loc[quality["usable"], "contrast"])
for label, sig in ((f"all {N}", m[m["q_value"] < 0.10]),
                   ("usable axes", m[(m["q_value"] < 0.10) & m["contrast"].isin(usable_set)]),
                   ("non-usable", m[(m["q_value"] < 0.10) & ~m["contrast"].isin(usable_set)])):
    print(f"sign agreement with the binary arm's logFC, {label:12s} ({len(sig):3d} hits): "
          f"{(np.sign(sig['logFC_within']) == np.sign(sig['logFC_binary'])).mean()*100:.0f}%")


In [ ]:
from collections import Counter
hits = coefficients[(coefficients["arm"] == "s_within_crispant") & (coefficients["q_value"] < 0.10)]
recurrence = Counter(hits["cell_type"])
observed = sum(v >= 3 for v in recurrence.values())

# Null: redistribute the same number of hits per contrast at random among the cell types actually
# tested in that contrast. This absorbs the fact that abundant types are tested everywhere.
rng = np.random.RandomState(0)
tested = (coefficients[coefficients["arm"] == "s_within_crispant"]
          .groupby("contrast")["cell_type"].apply(list))
per_contrast = hits.groupby("contrast").size()
null = []
for _ in range(2000):
    counter = Counter()
    for contrast, k in per_contrast.items():
        counter.update(rng.choice(tested[contrast], size=k, replace=False))
    null.append(sum(v >= 3 for v in counter.values()))
print(f"{len(recurrence)} distinct cell types hit across {len(hits)} hits")
print(f"hit in >=3 contrasts: observed {observed}, null mean {np.mean(null):.1f}, "
      f"p = {(np.array(null) >= observed).mean():.3f}")
display(pd.DataFrame(recurrence.most_common(8), columns=["cell_type", "n_contrasts"]))


**Finding 5 — the headline, split by how the gain arrives.**

Everything is measured against one fixed baseline: **Model 1, `~ binary`** — the analysis anyone
would run without morphology. Adding the within-group slope terms buys cell types through two
channels that deserve different amounts of trust.

| | all 35 | usable axes (16) |
|---|---|---|
| baseline `B1`, simple binary contrast | 532 | 331 |
| **indirect** — group contrast sharpened (`B2\B1\S`) | **+17** | **+12** |
| **direct** — slope hits the plain binary misses (`S\B1`) | **+67** | **+46** |
| total resolved (\|B1∪B2∪S\|) | **616** | **389** |
| gain over baseline | **+84 (16%)** | **+58 (18%)** |
| cost: lost from the binary coefficient | −7 | −3 |

The three segments are **disjoint and sum to the union**, so bar length is total cell types resolved.

**There is deliberately no "both" category.** `s_within` is orthogonalised to `binary` by
construction, so the slope is structurally barred from sharing credit for the group difference.
Intersecting the two hit sets understates the real overlap rather than measuring it. Overlap between
morphology and the label is §2b's question, where the predictor is the full `s` and nothing has been
orthogonalised.

**Direct is the discovery claim, and it has a matched negative control: 67 against 1.** Same embryos,
same model, same permutation machinery, same FDR threshold — only which group carries the gradient
changed. In the gated subset it is **46 against 0**.

**Indirect now has its own matched control too, and it passes.** These are not detections —
morphology did not find them. The slope columns absorb within-group scatter, the quasi-likelihood
dispersion falls, and the *group* contrast sharpens enough to carry borderline cell types over FDR.
That mechanism is generic to adding any informative covariate, so on its own it says nothing about
morphology. The test: two designs adding exactly **one** column each, identical degrees of freedom,
differing only in which group carries the gradient.

| one-extra-column design | additions | lost | net | contrasts |
|---|---|---|---|---|
| `~ binary + s_within_crispant` | **+37** | −7 | **+30** | 14 of 35 |
| `~ binary + s_within_control` *(negative control)* | +4 | −5 | **−1** | 4 of 35 |

Crispant beats control in **12 of 35** contrasts and **never the reverse** (Wilcoxon p = 0.002). An
equally sized column of *control-side* morphological variation buys nothing. So the sharpening is
driven by genuine within-crispant structure, not by the mere act of adding a covariate — indirect is
a real benefit of measuring morphology, just not a detection by it.

**What is still unresolved.** The conditional formulation is deliberately conservative: it charges
the group difference entirely to the label and lets the gradient claim only what is orthogonal to it.
§2b shows that the unconditional `s` arm resolves cell types the conditional one cannot, so this
design leaves power on the table. Which formulation is *correct* for discovery is not settled here —
and the difficulty is largely that FDR makes hit counts non-additive, since BH thresholds move with
the whole p-value distribution. Compared as coefficient vectors instead of hit sets (§2), the same
comparison is a smooth geometry problem with none of this discreteness.

Two supporting observations:

- **It behaves like a dose.** Among the 165 slope hits, the within-slope log fold change shares sign
  with the binary arm's in **91%** of cases (92% within the gated subset). Morphologically severe
  crispants deviate *further in the same direction* as the perturbation effect — a graded amount of
  the same thing, not a different axis. This is also what §2's high cosine predicts.
- **It is not a reproducible cell-type signature.** 65 distinct types are hit; 24 in three or more
  contrasts — but a null redistributing the same hits at random among tested types gives 25.3
  (p ~ 0.77). Which cell types respond does not recur across perturbations beyond what abundance
  explains.

Gains are concentrated, not spread: `atf6 | 34C | 36hpf` alone contributes 12 direct additions, and
20 of 35 contrasts gain nothing at all.

---

## What this supports, and what it does not

**Supported.** Within a crispant clutch, morphological severity predicts the *magnitude* of the
transcriptional phenotype — same direction, graded amount — resolving 67 cell-type effects the class
label alone misses, against 1 in a matched negative control. Restricting to the 16 contrasts whose
morphology axis was validated before the sequencing data was touched, it is 46 against 0. Where there
is graded morphology to exploit, morphology adds real information.

**Not supported.** That a graded morphology score should replace the class label — decisively against
on `T_std` (§1), though **not** on cell types resolved, where the two are indistinguishable among
validated axes (§2b). That morphology identifies *which* cell types are dose-sensitive (§5, recurrence
at chance). That morphological severity cleanly stratifies molecular severity (§3, directionally
right, weak).

**Open confound.** Contrasts with strong morphological phenotypes are plausibly those where the
perturbation simply *worked better*. Morphology and transcriptome would then be co-symptomatic of
editing efficiency rather than one being informative about the other, and every panel in §4 would look
identical either way. The doubled binary hit rate in the gated subset is exactly what that confound
predicts. Separating them requires an independent measure of editing efficiency — sequence the guides,
or genotype the F0s.

**Known soft spots, for audit.** One contrast (`atf6 | 34C | 30hpf`) is excluded for single-point
leverage — see §0; the call was made on the morphology distribution alone, but it does move the
NEW-rate comparison above. The 9/35 count in §4a flickers by ±1 with the permutation draw at 2000
permutations. Six metrics were compared in §4b and the best is quoted. The three-level fit in §3 is
estimable in only 21 of 35 contrasts. 18 embryos with morphology are absent from the CDS. FDR is
applied within each contrast, so the 67 figure is a sum over 35 independently-corrected families
rather than a single globally-corrected count. Cell types significant for both the slope and the sharpened binary coefficient (15) are
credited to `direct`; charging them to `indirect` instead gives 52 rather than 67. The union total
is 616 either way.


---

## 7. FDR sensitivity — does any of this depend on q < 0.10?

Every count above uses BH **q < 0.10**. BH is adaptive, so tightening the gate does not simply scale
all the buckets down: the threshold moves with the whole p-value distribution, and a bucket built
from a *set difference* between two arms can move in either direction. So the question is not
whether counts shrink — they must — but whether the **morphology share** survives.

Three count families are re-derived at **q < 0.05** below, with nothing refit: the same coefficient
tables, the same designs, only the gate moved.

1. **§2 head-to-head** — binary-only / shared / `s`-only, and `s`-only as a share of the union.
2. **§5 gain accounting** — baseline / indirect / direct, and (indirect + direct) as a share of the
   morphology-augmented total.
3. **§3 DACTs** — escaper and severe sub-contrasts.

In [ ]:
SENSITIVITY_Q = (0.10, 0.05)
rows = []

for threshold in SENSITIVITY_Q:
    # ---- 1. head-to-head, per morph arm, on all contrasts and on validated-axis contrasts only ----
    for arm in ("s_z", "hinge_z"):
        overlap = ep.overlap_counts(coefficients, coefficients, arm=arm, reference="binary_z",
                                    q_threshold=threshold, exclude=EXCLUDED_CONTRASTS)
        overlap = overlap.merge(quality[["contrast", "usable"]], on="contrast", how="left")
        overlap["usable"] = overlap["usable"].fillna(False).astype(bool)
        for label, block in (("all", overlap), ("validated", overlap[overlap["usable"]])):
            union = int(block[["reference_only", "shared", "arm_only"]].sum().sum())
            rows.append({
                "family": f"head-to-head {arm}", "subset": label, "q": threshold,
                "n_contrasts": len(block),
                "binary_only": int(block["reference_only"].sum()),
                "shared": int(block["shared"].sum()),
                "morph_only": int(block["arm_only"].sum()),
                "total": union,
                "morph_share": (int(block["arm_only"].sum()) / union) if union else np.nan,
            })

    # ---- 2. gain accounting (§5). Take the table slope_gain_bars builds, discard its figure. ----
    for label, usable_only in (("all", False), ("validated", True)):
        figure, gains = ep.slope_gain_bars(binary_check, quality, q_threshold=threshold,
                                           usable_only=usable_only)
        plt.close(figure)
        baseline = int(gains["baseline"].sum())
        indirect, direct = int(gains["indirect"].sum()), int(gains["direct"].sum())
        augmented = baseline + indirect + direct
        rows.append({
            "family": "gain accounting", "subset": label, "q": threshold,
            "n_contrasts": len(gains), "binary_only": baseline, "shared": np.nan,
            "morph_only": indirect + direct, "total": augmented,
            "morph_share": (indirect + direct) / augmented if augmented else np.nan,
            "indirect": indirect, "direct": direct,
            "lost": int(gains["lost"].sum()),
            "neg_control": int(gains["control_slope"].sum()),
        })

    # ---- 3. DACTs (§3) ----
    for arm in DACT_ARMS:
        rows.append({"family": f"DACT {arm}", "subset": "all", "q": threshold,
                     "n_contrasts": dact["contrast"].nunique(),
                     "morph_only": int((dact[f"q_{arm}"] < threshold).sum())})

sensitivity = pd.DataFrame(rows)
target = EDGER / "q_sensitivity.csv"
sensitivity.to_csv(target, index=False)

wide = (sensitivity.pivot_table(index=["family", "subset"], columns="q",
                                values=["morph_only", "total", "morph_share"])
        .reindex(columns=SENSITIVITY_Q, level=1))
display(wide.round(3))
print(f"\nwrote {target}")

In [ ]:
# Does the morphology SHARE hold up? Only the families where a share is defined.
share = sensitivity.dropna(subset=["morph_share"]).copy()
pivot = share.pivot_table(index=["family", "subset"], columns="q", values="morph_share")
pivot["change_pp"] = 100 * (pivot[0.05] - pivot[0.10])

print("morphology share of the total, q<0.10 -> q<0.05\n")
for (family, subset), row in pivot.iterrows():
    print(f"  {family:26s} {subset:10s} "
          f"{row[0.10]:6.1%} -> {row[0.05]:6.1%}   ({row['change_pp']:+.1f} pp)")

counts = sensitivity.pivot_table(index=["family", "subset"], columns="q", values="morph_only")
counts["retained"] = counts[0.05] / counts[0.10]
print("\nmorph-attributable cell types retained at the tighter gate\n")
for (family, subset), row in counts.iterrows():
    print(f"  {family:26s} {subset:10s} "
          f"{int(row[0.10]):4d} -> {int(row[0.05]):4d}   ({row['retained']:.0%} retained)")

figure, axes = plt.subplots(1, 2, figsize=(11.4, 4.4))
labels = [f"{f}\n({s})" for f, s in pivot.index]
y = np.arange(len(pivot))
for offset, threshold, colour in ((-0.19, 0.10, "0.62"), (0.19, 0.05, "#C0392B")):
    axes[0].barh(y + offset, 100 * pivot[threshold], height=0.36, color=colour,
                 label=f"q < {threshold:g}", zorder=3)
axes[0].set_yticks(y); axes[0].set_yticklabels(labels, fontsize=8.5)
axes[0].invert_yaxis(); axes[0].set_xlabel("morphology share of total (%)")
axes[0].legend(fontsize=9); axes[0].set_title("Share is what matters", fontsize=11)

keep = counts.dropna()
labels2 = [f"{f}\n({s})" for f, s in keep.index]
y2 = np.arange(len(keep))
axes[1].barh(y2, 100 * keep["retained"], height=0.6, color="#8FA9C4", zorder=3)
axes[1].axvline(100, color="0.55", linestyle="--", linewidth=1.1, zorder=2)
axes[1].set_yticks(y2); axes[1].set_yticklabels(labels2, fontsize=8.5)
axes[1].invert_yaxis(); axes[1].set_xlabel("morph counts retained at q<0.05 (% of q<0.10)")
axes[1].set_title("Absolute counts do shrink", fontsize=11)
figure.tight_layout()
ep.save_inline(figure, FIGURES / "7_q_sensitivity")